# Day 01 — Why AI Applications Need Evaluation

**Module 1 · Foundations**

Traditional software can often be tested by checking whether the output exactly matches the expected value.

LLM applications are different.

The same input can produce different valid outputs, which means that exact-match testing is often not enough.

In this notebook, we will:

1. See how traditional unit testing works.
2. Observe LLM output variability.
3. Understand why exact-match testing breaks down.
4. Introduce the AI evaluation problem.
5. Understand the evaluation loop.
6. See where DeepEval fits into the process.

> **Core idea:** We evaluate the *quality of behavior*, not just exact output equality.

## 1. Environment Check

Before building evaluations, let's verify that our evaluation framework is installed correctly.

In [1]:
import deepeval

print("DeepEval", deepeval.__version__)


DeepEval 4.2.1


## 2. Traditional Software Testing

Let's start with something deterministic.

A traditional function such as `add()` should return the same result every time for the same input.

This makes exact assertions straightforward.

In [2]:
def add(a: int, b: int) -> int:
    return a + b


result = add(2, 3)

assert result == 5

print("Test passed.")

Test passed.


### Why does this test work?

The behavior is deterministic:

```text
Input → Function → Exact Output

## 3. Let's Test an LLM

Now we will call an LLM with the same question more than once.

Our goal is not yet to evaluate it.

We simply want to **observe its behavior**.

In [3]:
import os

from dotenv import load_dotenv
from groq import Groq

load_dotenv()

assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found."

client = Groq()


def reply(prompt: str, temperature: float = 1.0) -> str:
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
    )

    return response.choices[0].message.content.strip()

In [4]:
question = "Explain what an LLM token is in one sentence."

answer_1 = reply(question)
answer_2 = reply(question)

print("Run 1:")
print(answer_1)

print("\nRun 2:")
print(answer_2)

Run 1:
A token is a single unit of text—such as a word, sub‑word piece, punctuation mark, or character—that an LLM processes and counts when encoding input and generating output.

Run 2:
An LLM token is a discrete unit of text—such as a word, subword, character, or punctuation mark—that the model reads and processes as a single element during generation and inference.


## 4. What Did We Observe?

The same input produced two responses.

They may not be textually identical.

But does that mean the model failed?

**No.**

Two different answers can both be correct.

This gives us the fundamental problem:

> **Exact string equality is usually not an appropriate definition of correctness for LLM applications.**

In [5]:
print("Are the responses exactly identical?")
print(answer_1 == answer_2)

Are the responses exactly identical?
False


## 5. From Exact Matching to Quality Evaluation

For deterministic software, we can often ask:

> "Did the program return exactly what I expected?"

For an LLM application, we often need to ask:

> "Is the response correct, relevant, useful, and appropriate?"

The testing model therefore changes.

### Traditional testing

```text
Input
  ↓
Program
  ↓
Expected Output
  ↓
Exact Comparison
  ↓
PASS / FAIL

## 6. The AI Evaluation Loop

A practical evaluation system can be understood through four core components.

### 1. Dataset

A collection of representative examples used to evaluate the application.

A test case may contain an input and, when available, an expected answer or reference context.

### 2. Metric

A method for judging the quality of the application's output.

Examples include correctness, relevance, faithfulness, and safety.

### 3. Threshold

The minimum acceptable score for a metric.

For example:

```text
Answer Relevancy ≥ 0.80